# B06 — Day 6: Optimizer ↔ Queue-Simulator Interface
**SupplyShield · Person B · Authoritative Day 6 notebook**

This notebook audits the actual optimizer and queue simulator, states their mathematical contracts, and validates a proposed per-OD plan interface on deterministic toy networks. The v1 material was reviewed; useful explanations, ownership boundaries, invariants, and toy results are consolidated here. This is the only authoritative Day 6 notebook. Execution outputs are saved in this notebook; experiment artifacts live separately under `experiments/day6/`.

**Scope and safety**
- No B01–B05 artifact is regenerated; no teammate-owned implementation is edited.
- The proposed `_routes` adapter runs in memory only and is restored in `finally`.
- No real 292-commodity optimizer or real-network queue run is performed.
- Existing experiment outputs are preserved; the toy test's original outputs were redirected to the Day 6 experiment area.

**Evidence tags:** `[VERIFIED]` means checked from current files/hashes; `[READ]` means read from source; `[TOY]` means demonstrated on a toy graph; `[PROPOSED]` means not implemented or agreed; `[OPEN]` means a decision or evidence is still needed.


In [1]:
import sys, math, json, platform
sys.dont_write_bytecode = True          # importing repo modules must NOT create __pycache__ files in your repo
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, networkx as nx, scipy
from scipy.optimize import linprog

def find_repo(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "data/processed/core_edges.csv").exists() and (p / "sim/simulator.py").exists():
            return p
    return None

REPO = find_repo()
if REPO:
    sys.path.insert(0, str(REPO))
print("repo found      :", REPO is not None, f"({REPO.name})" if REPO else "(sections that need the repo will be skipped)")
print("python/numpy/scipy/networkx/pandas:", platform.python_version(), np.__version__, scipy.__version__, nx.__version__, pd.__version__)

repo found      : True (supplyshield--1)
python/numpy/scipy/networkx/pandas: 3.10.8 2.2.6 1.15.3 3.4.2 2.3.3


## 1. Repository, artifacts, and ownership `[VERIFIED]`

Repository: `C:\Local\Documents-Local\ML Project\supplyshield--1`. At this audit, branch `person-b-prediction-cascade` was at `0b4e435b462ed67ca683077b9f64e5835817f91f`, equal to the local `origin/person-b-prediction-cascade` tracking ref. No fetch was performed; live remote state is unverified.

- B04 `experiments/day4/final_v2/run_manifest_v1.json`: recorded input and output hashes match the current files. The manifest includes `sim/simulate.py`, `docs/assumptions.md`, and `docs/cascade_rules_v1.md`.
- B05 master-split risk table: 296 rows; its hash matches the manifest and success marker. It represents representative-leg scores, not lane-aggregate coverage.
- B02 master-split model, card, and saved calibration/test predictions are present. Legacy result cells 13, 19, 60, and 63 still write superseded comparison JSONs; do not run those cells as part of this Day 6 work.
- B03 master-split Method C parameters, test intervals, run manifest, and success marker are present.
- The `.venv` has Python 3.10.8 with NumPy 2.2.6, SciPy 1.15.3, NetworkX 3.4.2, and pandas 2.3.3. Its `nbformat` module is unavailable; no package was installed.

**Ownership boundaries** (from project handoff and existing ownership notes):

| Area | Files | Owner boundary |
|---|---|---|
| Prediction, uncertainty, B04 structural cascade, B05 risk table | B01–B05 notebooks, `models/`, `experiments/day2–5/`, `sim/simulate.py` | Person B |
| Multi-flow optimizer | `opt/c_multiflow.py`, `opt/c_real_graph.py`, `opt/c_toy_recovery.py`, `docs/c_notes.md` | Person C; review required before edits |
| Data/graph and daily queue simulator | `src/`, core datasets, `sim/simulator.py`, `sim/test_core.py`, `sim/test_toy.py` | Person A/Jahnavi; review required before edits |

**B04 lineage constraint:** `sim/simulate.py` is hashed by the B04 final_v2 manifest. It has not been changed; editing it would invalidate that recorded lineage unless B04 were rerun/versioned.


## 2. What the optimizer actually solves  `[READ]` — `opt/c_multiflow.py::solve_multiflow(G, closed=None)`

**Simple version.** Every lane of the core graph is treated as its own "commodity": legs that want to go from hub *u* to hub *v*. In normal times they use the direct lane. The LP decides, *for every commodity separately*, which lanes to use so that total travel hours + a penalty for undelivered legs is minimal, while all commodities **share** each hub's dispatch capacity.

**Math.** Hubs $N$ (100), directed lanes $E$ (292), commodities $K$ (one per lane, $k=(o_k,d_k)$), daily demand $q_k=\text{trips}_k/14$ legs/day.

$$\min_{x,u}\;\sum_{k\in K}\sum_{e\in E} c_e\,x_{k,e}\;+\;P\sum_{k\in K}u_k \qquad (P=\texttt{PENALTY\_HOURS}=96,\ c_e=\text{median lane hours})$$

subject to

$$\sum_{e\in\text{out}(n)}x_{k,e}-\sum_{e\in\text{in}(n)}x_{k,e}=\begin{cases}q_k-u_k & n=o_k\\-(q_k-u_k) & n=d_k\\0&\text{otherwise}\end{cases}\qquad\forall k,n \quad\text{(per-commodity conservation, lines 59–73)}$$

$$\sum_{k}\sum_{e\in\text{out}(n)}x_{k,e}\le \text{cap}_n\ \ \forall n\quad\text{(shared hub dispatch, lines 75–77)}\qquad 0\le u_k\le q_k,\ \ x\ge 0$$

and, if a lane $e^\*$ is closed: $x_{k,e^\*}=0\ \forall k$ (lines 81–82).

**What it returns (line 90):** only `(total_cost, unmet)` — two scalars. The solution vector `res.x` holds every $x_{k,e}$ and $u_k$ but is discarded. **The per-commodity plan exists inside the function and is thrown away.**

**Properties that matter for the interface**
- Steady-state **rates** (legs/day). There is no time index, no queue, no buffer.
- Capacity constrains **outgoing** legs only; edge capacity is `BIG` (not modelled).
- `closed` is **one lane**. There is no hub-closure input.
- Costs and capacities are in **hours** and **legs/day**, from `core_edges.median_actual_time / 60` and `core_nodes.capacity_est`.

In [2]:
# Evidence: the exact source lines, so you can see them yourself (read-only).
def show(path, a, b):
    lines = (REPO / path).read_text(encoding="utf-8", errors="replace").splitlines()
    for i in range(a - 1, b):
        print(f"{path}:{i+1:>3} | {lines[i].rstrip()}")
if REPO:
    show("opt/c_multiflow.py", 88, 90)
    print()
    show("opt/c_multiflow.py", 81, 82)
    print()
    show("sim/simulator.py", 27, 32)

opt/c_multiflow.py: 88 |     unmet = float(res.x[K * E:].sum())
opt/c_multiflow.py: 89 |     travel = float(np.dot(np.tile(cost, K), res.x[:K * E]))
opt/c_multiflow.py: 90 |     return travel + rg.PENALTY_HOURS * unmet, unmet

opt/c_multiflow.py: 81 |     if closed is not None:
opt/c_multiflow.py: 82 |         bounds_hi[np.arange(K) * E + eidx[closed]] = 0.0

sim/simulator.py: 27 |     # Aggregate edge flows cannot identify multiple OD commodities.
sim/simulator.py: 28 |     if len(demands) != 1:
sim/simulator.py: 29 |         raise ValueError(
sim/simulator.py: 30 |             "An edge-flow plan currently supports one origin/destination. "
sim/simulator.py: 31 |             "Multiple commodities require a commodity-specific plan."
sim/simulator.py: 32 |         )


## 3. What the daily queue simulator expects  `[READ]` — `sim/simulator.py::simulate(graph, scenario, plan=None)`

**Simple version.** Parcels are generated every day for `demand_days` days. Each parcel batch follows a **fixed path**. Each hub can *dispatch* at most `cap` legs per day (times a failure multiplier); every lane hop takes **one day**; a hub can only hold `buffer` legs (default $2\max(\text{cap},1)$). Whatever cannot move waits in a queue.

| Aspect | Behaviour (line refs) |
|---|---|
| Time | Daily steps up to `horizon`; one lane = one day (L119, L222–227) |
| Demand | `graph.graph["demands"] = {(o,d): legs/day}`, released for `demand_days` days (L124, L229–235) |
| Routes | `plan=None` → `nx.shortest_path` by `cost` for each OD (L14–25). With a plan → **exactly one OD**, edge-flow dict decomposed into paths (L27–102) |
| Failures | Only **hub** capacity multipliers `scenario["failed"] = {hub: remaining fraction}` for `failure_days` (L137–153). **No lane-closure field.** Lanes can only be limited through a static `edge["cap"]` |
| Dispatch | Per hub per day $\le \text{cap}\times\text{scale}\times\text{multiplier}$, per lane $\le$ `edge cap`, and receiving hub needs buffer room (L274–293) |
| Costs | `shipping` = Σ sent × lane cost (L293); `waiting` = `waiting_cost` × (queued + outside backlog) **every day** (L312); `total = shipping + waiting + unmet_penalty × unmet` (L364) |
| Unmet | `generated − delivered` at the end (L345); unplanned demand stays "outside" forever |
| Safety | Asserts flow conservation every day (L318) and buffer limits (L329) |

**Good news:** after `_routes()` returns a list of `(path, amount)`, *everything downstream already handles any number of commodities* — the queues carry `[path, position, amount]` jobs and never look at commodity identity. The multi-OD limitation is **only** the `len(demands) != 1` guard and the single-OD decomposition inside `_routes()`.

## 4. The integration gap, itemised  `[READ]` unless marked

| # | Gap | Severity | Evidence |
|---|---|---|---|
| G1 | Optimizer **discards** its per-commodity flows $x_{k,e}$ | **Blocker** | `solve_multiflow` returns `(cost, unmet)` only |
| G2 | Simulator **rejects** any edge-flow plan with ≠ 1 demand pair | **Blocker** | `simulator.py` L27–32; confirmed by executing it in §7 `[TOY]` |
| G3 | Closures: optimizer = one lane; simulator = hub multipliers only, lane closure only via a permanent static edge `cap` | Must define | L81–82 vs L137–153 |
| G4 | Time semantics: LP gives daily **rates**; simulator is time-stepped with 1-day hops, queues, buffers | Must define | contract below treats a plan as *per-day release rates, applied every day* |
| G5 | **Demand source differs** between optimizer and simulator inputs | Must fix | §5 `[VERIFIED]`: 28 of 292 lanes, 38 legs |
| G6 | Penalty value: code 96 h; `docs/c_notes.md` says 48 h; simulator default 20 (its test sets 96) | Must pin | `c_real_graph.py` L41, `c_notes.md`, `simulator.py` L185 |
| G7 | Objectives differ: LP has no waiting cost; simulator adds `waiting_cost` per leg-day | Must define | compare **shipping + penalty** for parity; report waiting separately |
| G8 | Waiting cost on permanently-unmet legs grows with the horizon, so `total_cost` is horizon-dependent whenever unmet > 0 | Note | `[TOY]` §7: 2,160 → 4,560 → 9,360 for horizon 20/40/80 |
| G9 | A closed hub as a *destination*: LP can still deliver into it; simulator blocks it | Edge case | `simulator.py` L282; LP only caps *outgoing* flow |
| G10 | Zero-capacity destination-only hubs (3 in the core): **your** `simulate_scenario` treats cap as a limit on *deliveries*, the LP/queue simulator do not | Must define | `[TOY]` §7: all 12 legs reported unmet |
| G11 | Packaging: optimizer uses bare imports (`import c_real_graph`), no `opt/__init__.py`; sim tests import `opt.c_toy_recovery` | Practical | `c_multiflow.py` L31–32 |
| G12 | Numerical noise: LP flows ≈ 1e-9–1e-7 vs simulator tolerances 1e-9 (EPS) and 1e-6 | Practical | needs thresholding + validation |
| G13 | "Do nothing" is undefined for the simulator: `plan=None` routes by *shortest path ignoring capacity and closures* | Must define | §7 shows it congests even with no failure |

Cost parity was also checked and is **not** a gap: all 292 lane costs from `core_edges.csv` equal the simulator test's recomputed completed-leg medians (see §5).

## 5. Read-only data audit: do the two sides even see the same inputs?  `[VERIFIED]`

The cell below only **reads** three CSVs and compares the optimizer's demand/cost inputs with the simulator test's inputs. It writes nothing.

In [3]:
if REPO:
    P = REPO / "data/processed"
    edges = pd.read_csv(P / "core_edges.csv"); nodes = pd.read_csv(P / "core_nodes.csv"); legs = pd.read_csv(P / "core_train_legs.csv")
    START, CUTOFF = pd.Timestamp("2018-09-12"), pd.Timestamp("2018-09-26")
    cr = pd.to_datetime(legs.trip_creation_time, format="mixed"); dp = pd.to_datetime(legs.od_start_time, format="mixed"); ar = pd.to_datetime(legs.od_end_time, format="mixed")
    in_win = cr.ge(START) & cr.lt(CUTOFF) & dp.ge(START) & dp.lt(CUTOFF)
    act = pd.to_numeric(legs.actual_time_min, errors="coerce")
    done = in_win & ar.ge(dp) & ar.lt(CUTOFF) & np.isfinite(act) & act.gt(0)
    K = ["source_center", "destination_center"]

    # Demand: optimizer uses edges.trips/14 ; simulator test (and B04) use in-window departures/14
    sim_dem = legs[in_win].groupby(K).size(); sim_dem.index.names = ["source", "destination"]
    ed = edges.set_index(["source", "destination"])
    q = pd.concat([sim_dem.rename("sim_legs"), ed.trips.rename("optimizer_legs")], axis=1)
    diff = q[q.sim_legs != q.optimizer_legs]
    print(f"hubs={len(nodes)}  lanes={len(edges)}  zero-capacity hubs={(nodes.capacity_est==0).sum()}  total cap={nodes.capacity_est.sum():.0f} legs/day")
    print(f"optimizer daily demand : {q.optimizer_legs.sum():>5} legs / 14 = {q.optimizer_legs.sum()/14:.4f} legs/day")
    print(f"simulator daily demand : {q.sim_legs.sum():>5} legs / 14 = {q.sim_legs.sum()/14:.4f} legs/day  (B04 uses the same 5,357)")
    print(f"lanes where they differ: {len(diff)} of {len(q)}; extra legs in optimizer input = {int(q.optimizer_legs.sum()-q.sim_legs.sum())}")
    print(f"legs in core_train_legs outside the Sep-12 window: {int((~in_win).sum())}  <- the source of the 38")
    print(diff.head(5).to_string())

    # Cost: optimizer uses edges.median_actual_time/60 ; simulator test recomputes completed-leg medians/60
    med = legs.loc[done].groupby(K).actual_time_min.median(); med.index.names = ["source", "destination"]
    c = pd.DataFrame({"optimizer_min": ed.median_actual_time}).join(med.rename("sim_min"))
    print(f"\ncost: lanes with different median minutes = {(c.optimizer_min - c.sim_min).abs().gt(1e-9).sum()} of {len(c)}  (cost source is consistent)")

    # Penalty wording across files (text search only; modules are NOT imported)
    import re
    for f in ["opt/c_real_graph.py", "docs/c_notes.md", "sim/test_core.py"]:
        txt = (REPO / f).read_text(encoding="utf-8", errors="replace")
        print(f"{f:22s}", re.findall(r"(PENALTY_HOURS\s*=\s*\d+\.?\d*|unmet_penalty\"\]\s*=\s*\d+\.?\d*|set to \*\*\d+ hours)", txt)[:3])
else:
    print("repo not found - skipped")

hubs=100  lanes=292  zero-capacity hubs=3  total cap=557 legs/day
optimizer daily demand :  5395 legs / 14 = 385.3571 legs/day
simulator daily demand :  5357 legs / 14 = 382.6429 legs/day  (B04 uses the same 5,357)
lanes where they differ: 28 of 292; extra legs in optimizer input = 38
legs in core_train_legs outside the Sep-12 window: 38  <- the source of the 38
                           sim_legs  optimizer_legs
source       destination                           
IND000000ACA IND160002AAC        57              59
IND000000ACB IND160002AAC        37              38
             IND501359AAE        18              19
             IND562132AAA        40              41
             IND712311AAA        31              32

cost: lanes with different median minutes = 0 of 292  (cost source is consistent)
opt/c_real_graph.py    ['PENALTY_HOURS = 96.0']
docs/c_notes.md        ['set to **48 hours', 'PENALTY_HOURS = 48']
sim/test_core.py       ['unmet_penalty"] = 96.0']


**Reading of the result.** Costs are consistent. Demand is *almost* consistent: 38 legs that departed before the 12 Sep window start are counted by the optimizer (`edges.trips`, 5,395) but not by the simulator test or B04 (5,357). It is small (0.7%), but a plan optimised for 385.36 legs/day and evaluated against 382.64 legs/day will fail any strict "delivered + unmet = demand" check. **The contract must therefore carry the demand vector explicitly**, and the evaluator must refuse a plan whose demand differs from its own.

## 6. Proposed plan contract `[PROPOSED]`

Reuse the existing OD-edge record shape `{"od": [o,d], "edge": [u,v], "flow": legs_per_day}` supported by `sim/simulate.py::_normalize_plan`. A plan envelope carries the explicit demand vector, per-OD edge flows, unmet rates, closed lanes, units, and provenance. Internally the queue adapter needs `{(o,d): {(u,v): flow}}`; `sim/simulator.py::_routes` currently accepts a planned edge map for only one OD and rejects multiple ODs. The record list is not directly accepted by the queue simulator.

| Field | Type / unit | Meaning |
|---|---|---|
| `demands` | `{(o,d): legs/day}` | Explicit single source of truth; must equal `graph.graph["demands"]` |
| `flows` | `{(o,d): {(u,v): legs/day}}` | Per-commodity edge allocations, applied as daily release rates |
| `unmet` | `{(o,d): legs/day}` | Demand intentionally left unserved by the optimizer |
| `closed_lanes` | `[(u,v), …]` | Directed lanes closed for the optimizer plan |
| `units` | `{flow: "legs/day", cost: "hours", penalty_per_unmet_leg: P}` | Pins accounting units and penalty |
| `provenance` | solver, version, objective, input hashes | Reproducibility metadata |

**Plan invariants**

| ID | Required check |
|---|---|
| I1 | `demands`, `flows`, `unmet`, and simulator graph have identical OD keys; plan demands equal graph demand |
| I2 | Every flow is finite, nonnegative, and uses an existing directed edge; sub-`1e-9` solver noise is dropped before serialization |
| I3 | Flow is conserved per OD at every intermediate node, within `1e-6` |
| I4 | Delivered flow plus unmet demand equals requested demand per OD, within `1e-6` |
| I5 | All ODs together respect shared outgoing hub capacity and any finite shared edge capacity |
| I6 | No positive flow uses a listed closed lane |
| I7 | Each OD flow decomposes into source-to-destination paths with no leftover cycles or disconnected flow |

Failed hubs are a separate scenario concern: the current optimizer has no failed-hub input. The queue simulator applies a temporary hub multiplier and blocks transfers into and out of the failed node while active; those behaviors must not be represented as if optimizer support already exists.

The LP minimizes daily travel hours plus a fixed unmet penalty. Queue total cost also includes waiting cost accrued over simulated days. Compare shipping and unmet components separately unless demand, penalty, time units, and horizon are aligned.


In [4]:
# Contract helpers for toy validation only; no repository implementation is changed.
EPS, TOL, PENALTY = 1e-9, 1e-6, 96.0
from sim.simulator import _routes as _single_od_routes

def validate_plan(G, plan, tol=TOL):
    """Validate OD conservation, shared capacities, closures and path decomposition."""
    required = {"demands", "flows", "unmet", "closed_lanes", "units", "provenance"}
    if not isinstance(plan, dict) or not required.issubset(plan):
        raise ValueError("plan is missing required contract fields")
    graph_demands = G.graph.get("demands", {})
    demands, flows, unmet = plan["demands"], plan["flows"], plan["unmet"]
    if set(demands) != set(graph_demands) or set(flows) != set(graph_demands) or set(unmet) != set(graph_demands):
        raise ValueError("I1: demands, flows, unmet and graph demand must have identical OD keys")
    for od, graph_q in graph_demands.items():
        q = demands[od]
        if not math.isfinite(float(q)) or q < 0 or abs(float(q) - float(graph_q)) > tol:
            raise ValueError(f"I1: plan demand for {od} differs from simulator graph demand")
    units = plan["units"]
    if units.get("flow") != "legs/day" or units.get("cost") != "hours":
        raise ValueError("contract units must be legs/day and hours")
    penalty = units.get("penalty_per_unmet_leg")
    if penalty is None or not math.isfinite(float(penalty)) or float(penalty) < 0:
        raise ValueError("penalty_per_unmet_leg must be finite and nonnegative")
    if not isinstance(plan["provenance"], dict) or not plan["provenance"].get("solver"):
        raise ValueError("provenance.solver is required")

    closed = {tuple(edge) for edge in plan["closed_lanes"]}
    if any(not G.has_edge(*edge) for edge in closed):
        raise ValueError("I6: closed_lanes contains an unknown directed edge")
    hub_load = defaultdict(float)
    edge_load = defaultdict(float)
    for od, edge_flows in flows.items():
        if not isinstance(edge_flows, dict):
            raise ValueError(f"I2: flows for {od} must be an edge-to-rate mapping")
        origin, destination = od
        if origin not in G or destination not in G or origin == destination:
            raise ValueError(f"I1: invalid OD endpoints {od}")
        balance = defaultdict(float)
        for edge, raw_flow in edge_flows.items():
            edge = tuple(edge)
            flow = float(raw_flow)
            if len(edge) != 2 or not G.has_edge(*edge):
                raise ValueError(f"I2: unknown directed edge {edge}")
            if not math.isfinite(flow) or flow < 0:
                raise ValueError("I2: edge flows must be finite and nonnegative")
            if edge in closed and flow > tol:
                raise ValueError(f"I6: flow on closed lane {edge}")
            balance[edge[0]] += flow
            balance[edge[1]] -= flow
            hub_load[edge[0]] += flow
            edge_load[edge] += flow
        for node, net in balance.items():
            if node not in (origin, destination) and abs(net) > tol:
                raise ValueError(f"I3: commodity {od} is not conserved at {node}")
        delivered = balance[origin]
        if delivered < -tol or abs(balance[destination] + delivered) > tol:
            raise ValueError(f"I3: invalid source/destination balance for {od}")
        raw_unmet = unmet[od]
        if not math.isfinite(float(raw_unmet)) or raw_unmet < 0 or raw_unmet > float(demands[od]) + tol:
            raise ValueError(f"I4: unmet for {od} must be finite and between zero and demand")
        if abs(delivered + float(raw_unmet) - float(demands[od])) > tol:
            raise ValueError(f"I4: delivered plus unmet does not equal demand for {od}")
        # Reuse the actual queue simulator's current single-OD decomposer.
        _single_od_routes(G, {od: float(demands[od])}, edge_flows)

    for node in G:
        cap = float(G.nodes[node]["cap"])
        if not math.isfinite(cap) or cap < 0:
            raise ValueError(f"I5: invalid hub capacity at {node}")
        if hub_load[node] > cap + tol:
            raise ValueError(f"I5: shared hub dispatch exceeds capacity at {node}")
    for edge in G.edges:
        cap = float(G.edges[edge].get("cap", math.inf))
        if math.isnan(cap) or cap < 0:
            raise ValueError(f"I5: invalid edge capacity at {edge}")
        if edge_load[edge] > cap + tol:
            raise ValueError(f"I5: shared edge flow exceeds capacity at {edge}")
    return True

def to_records(plan):
    """Serialize nonzero flows for sim.simulate._normalize_plan; keep metadata separately."""
    return [{"od": list(od), "edge": list(edge), "flow": float(flow)}
            for od, edge_flows in plan["flows"].items()
            for edge, flow in edge_flows.items() if flow > EPS]

def decompose_one(graph, od, demand, flows):
    """Call the real queue simulator's existing single-OD route decomposition."""
    return _single_od_routes(graph, {od: float(demand)}, flows)

print("contract helpers defined: validate_plan, to_records, decompose_one")


contract helpers defined: validate_plan, to_records, decompose_one


## 7. Toy proof on a 5-hub network  `[TOY]`

Two commodities (A→D, B→D, 6 legs/day each) share hub **H1** (dispatch cap 8). Costs are hours. D is destination-only (cap 0), like three real core hubs.

```
        A ──2──► H1 ──2──► D          direct lanes: A→D = 6 h, B→D = 6 h
        B ──3──►/                     via H2: A→H2 3, B→H2 3.5, H2→D 4   (H2 cap 5)
```
Three scenarios: **S0** no closure · **S1** lane H1→D closed · **S2** H1→D *and* A→D closed (forces an unmet leg).

The LP below is the **same formulation as §2** (shared hub capacity, per-commodity conservation, `unmet ≤ q`), generalised to a *set* of closed lanes, and — the only real difference — it **exports** $x_{k,e}$.

In [5]:
def toy_graph():
    G = nx.DiGraph()
    for n, c in dict(A=10, B=10, H1=8, H2=5, D=0).items(): G.add_node(n, cap=float(c))
    for u, v, c in [("A","H1",2), ("H1","D",2), ("B","H1",3), ("A","H2",3), ("B","H2",3.5), ("H2","D",4), ("A","D",6), ("B","D",6)]:
        G.add_edge(u, v, cost=float(c), cap=math.inf)
    G.graph["demands"] = {("A","D"): 6.0, ("B","D"): 6.0}
    G.graph["waiting_cost"] = 24.0; G.graph["unmet_penalty"] = PENALTY
    return G

def solve_plan(G, closed=(), penalty=PENALTY):
    # Toy LP aligned with opt/c_multiflow's formulation. The production optimizer is not
    # called here because it does not return its flow variables and is real-network sized.
    nodes = list(G.nodes); ni = {n: i for i, n in enumerate(nodes)}
    edges = list(G.edges); ei = {e: i for i, e in enumerate(edges)}
    N, E = len(nodes), len(edges); ods = list(G.graph["demands"]); K = len(ods)
    q = np.array([G.graph["demands"][k] for k in ods]); nv = K * E + K
    Aeq = np.zeros((K * N, nv)); beq = np.zeros(K * N)
    for k, (o, d) in enumerate(ods):
        for e, (u, v) in enumerate(edges):
            Aeq[k*N + ni[u], k*E + e] += 1; Aeq[k*N + ni[v], k*E + e] -= 1
        Aeq[k*N + ni[o], K*E + k] += 1; Aeq[k*N + ni[d], K*E + k] -= 1
        beq[k*N + ni[o]] += q[k]; beq[k*N + ni[d]] -= q[k]
    Aub = np.zeros((N, nv))
    for k in range(K):
        for e, (u, v) in enumerate(edges): Aub[ni[u], k*E + e] = 1
    bub = np.array([G.nodes[n]["cap"] for n in nodes])
    hi = np.full(nv, np.inf); hi[K*E:] = q
    for k in range(K):
        for c in closed: hi[k*E + ei[tuple(c)]] = 0
    cost = np.concatenate([np.tile([G.edges[e]["cost"] for e in edges], K), np.full(K, penalty)])
    r = linprog(cost, A_ub=Aub, b_ub=bub, A_eq=Aeq, b_eq=beq, bounds=np.column_stack([np.zeros(nv), hi]), method="highs")
    assert r.success, r.message
    x = r.x
    flows = {od: {edges[e]: float(x[k*E + e]) for e in range(E) if x[k*E + e] > EPS} for k, od in enumerate(ods)}   # I2: drop noise
    unmet = {od: float(x[K*E + k]) if x[K*E + k] > EPS else 0.0 for k, od in enumerate(ods)}
    travel = sum(G.edges[e]["cost"] * f for fl in flows.values() for e, f in fl.items())
    return dict(demands=dict(G.graph["demands"]), flows=flows, unmet=unmet, travel=travel,
                total=travel + penalty * sum(unmet.values()), closed_lanes=[tuple(c) for c in closed],
                units={"flow": "legs/day", "cost": "hours", "penalty_per_unmet_leg": float(penalty)},
                provenance={"solver": "scipy.optimize.linprog(method=highs), toy formulation", "version": scipy.__version__})

G = toy_graph()
SCEN = {"S0": [], "S1": [("H1","D")], "S2": [("H1","D"), ("A","D")]}
plans = {n: solve_plan(G, c) for n, c in SCEN.items()}
rows = []
for n, p in plans.items():
    for od, fl in p["flows"].items():
        rows.append({"scenario": n, "commodity": f"{od[0]}→{od[1]}", "flows (legs/day)": ", ".join(f"{u}→{v}: {f:g}" for (u, v), f in fl.items()), "unmet": p["unmet"][od]})
print(pd.DataFrame(rows).to_string(index=False))
print("\nLP objective per day (travel hours | + 96 h × unmet):", {n: (round(p["travel"], 2), round(p["total"], 2)) for n, p in plans.items()})
for n, p in plans.items(): validate_plan(G, p)
print("validate_plan (I1–I6): all three plans pass")

scenario commodity         flows (legs/day)  unmet
      S0       A→D         A→H1: 6, H1→D: 6    0.0
      S0       B→D B→H1: 2, B→D: 4, H1→D: 2    0.0
      S1       A→D                   A→D: 6    0.0
      S1       B→D                   B→D: 6    0.0
      S2       A→D         A→H2: 5, H2→D: 5    1.0
      S2       B→D                   B→D: 6    0.0

LP objective per day (travel hours | + 96 h × unmet): {'S0': (58.0, 58.0), 'S1': (72.0, 72.0), 'S2': (71.0, 167.0)}
validate_plan (I1–I6): all three plans pass


In [6]:
# Prove the validator actually rejects bad plans (a validator that never fails proves nothing).
import copy
bad = {}
p = copy.deepcopy(plans["S0"]); p["flows"][("A","D")][("H1","D")] += 1.0;            bad["leak at H1 (I3)"] = p
p = copy.deepcopy(plans["S0"]); p["unmet"][("A","D")] = 2.0;                         bad["delivered+unmet≠demand (I4)"] = p
p = copy.deepcopy(plans["S0"]); p["flows"][("A","D")] = {("A","H1"): 12.0, ("H1","D"): 12.0}; p["unmet"][("A","D")] = 0; bad["demand 12≠6 (I4)"] = p
p = copy.deepcopy(plans["S0"]); p["flows"][("B","D")] = {("B","H1"): 6.0, ("H1","D"): 6.0};      bad["H1 over cap 8 (I5)"] = p
p = copy.deepcopy(plans["S1"]); p["closed_lanes"] = [("H1","D")]; p["flows"][("A","D")] = {("A","H1"): 6.0, ("H1","D"): 6.0}; bad["flow on closed lane (I6)"] = p
p = copy.deepcopy(plans["S0"]); del p["flows"][("B","D")];                           bad["missing commodity (I1)"] = p
for name, pl in bad.items():
    try: validate_plan(G, pl); print(f"NOT CAUGHT  : {name}")
    except ValueError as e: print(f"caught      : {name:32s} -> {e}")

caught      : leak at H1 (I3)                  -> I3: commodity ('A', 'D') is not conserved at H1
caught      : delivered+unmet≠demand (I4)      -> I4: delivered plus unmet does not equal demand for ('A', 'D')
caught      : demand 12≠6 (I4)                 -> I4: delivered plus unmet does not equal demand for ('A', 'D')
caught      : H1 over cap 8 (I5)               -> I5: shared hub dispatch exceeds capacity at H1
caught      : flow on closed lane (I6)         -> I6: flow on closed lane ('H1', 'D')
caught      : missing commodity (I1)           -> I1: demands, flows, unmet and graph demand must have identical OD keys


### 7a. What the current simulator does with a multi-OD plan, and the minimal in-memory patch

First the **unpatched** real `simulate()` (read-only call). Then I replace only `_routes` *in memory* with a version that loops the **verbatim existing decomposition** over commodities. The original is restored in a `finally` block; no file is touched.

In [7]:
if REPO:
    import sim.simulator as S
    _orig_routes = S._routes

    def routes_proposed(graph, demands, plan):
        # [PROPOSED] Reuse the real single-OD decomposer for every commodity.
        if plan is None:
            return _orig_routes(graph, demands, plan)
        nested = isinstance(plan, dict) and (not plan or isinstance(next(iter(plan.values())), dict))
        if len(demands) == 1 and not nested:
            return _orig_routes(graph, demands, plan)  # legacy single-OD compatibility
        normalized = {od: {} for od in demands}
        if isinstance(plan, list):
            for record in plan:
                od, edge = tuple(record["od"]), tuple(record["edge"])
                normalized.setdefault(od, {})[edge] = normalized.setdefault(od, {}).get(edge, 0.0) + float(record["flow"])
        elif nested:
            normalized = {tuple(od): {tuple(edge): float(flow) for edge, flow in edge_flows.items()}
                          for od, edge_flows in plan.items()}
        else:
            raise ValueError("multi-OD plans must be nested mappings or OD-edge-flow records")
        if set(normalized) != set(demands):
            raise ValueError("commodity plan must contain exactly the demand pairs")
        routes = []
        for od, amount in demands.items():
            routes.extend(decompose_one(graph, od, amount, normalized[od]))
        return routes

    def run(G, scn, plan, closed=()):
        H = G.copy()
        for lane in closed:
            H.edges[tuple(lane)]["cap"] = 0.0
        return S.simulate(H, scn, plan)

    SCN = dict(demand_days=5, horizon=40, failed={}, failure_days=0)

    try:
        run(G, SCN, plans["S0"]["flows"])
        raise AssertionError("current simulate() unexpectedly accepted a multi-OD plan")
    except ValueError as exc:
        print("CURRENT simulate() rejects multi-OD plans:", exc)
else:
    print("repo not found - skipped")


CURRENT simulate() rejects multi-OD plans: An edge-flow plan currently supports one origin/destination. Multiple commodities require a commodity-specific plan.


In [8]:
if REPO:
    results = {}
    S._routes = routes_proposed                       # in-memory only
    try:
        for n, p in plans.items():
            results[n] = run(G, SCN, p["flows"], SCEN[n])
        # reference policies on the SAME graph and SAME closed lane
        results["plan=None, no closure (S0)"]  = run(G, SCN, None, [])
        results["plan=None, H1→D closed (S1)"] = run(G, SCN, None, SCEN["S1"])
        # horizon dependence of waiting cost under permanent unmet (G8)
        hz = {h: run(G, dict(SCN, horizon=h), plans["S2"]["flows"], SCEN["S2"])["waiting_cost"] for h in (20, 40, 80)}
        # capacity stress: halve every hub's dispatch
        stress = run(G, dict(SCN, capacity_scale=0.5), plans["S0"]["flows"], [])
        # Full hub failure: queue simulator applies zero dispatch during failure days.
        GF = nx.DiGraph(); GF.add_node("A", cap=10.0, buffer=4.0); GF.add_node("D", cap=0.0, buffer=2.0)
        GF.add_edge("A", "D", cost=1.0, cap=math.inf); GF.graph["demands"] = {("A", "D"): 2.0}
        failure_result = S.simulate(GF, dict(demand_days=1, horizon=5, failure_start=0, failure_days=2, failed={"A": 0.0}), None)
        # legacy single-OD compatibility (T1)
        G1 = toy_graph(); G1.graph["demands"] = {("A","D"): 6.0}
        p1 = solve_plan(G1, [("H1","D")])
        legacy = _orig_routes(G1, G1.graph["demands"], p1["flows"][("A","D")])
        nested = routes_proposed(G1, G1.graph["demands"], p1["flows"])
    finally:
        S._routes = _orig_routes                      # ALWAYS restore
    assert S._routes is _orig_routes
    print("in-memory patch removed; original sim.simulator._routes restored:", S._routes is _orig_routes)

    tab = pd.DataFrame({k: {"generated": r["generated"], "delivered": r["delivered"], "unmet": r["unmet"], "shipping": r["shipping_cost"],
                            "waiting": r["waiting_cost"], "total": r["total_cost"], "recovery_days": r["recovery_days"]} for k, r in results.items()}).T.round(2)
    print(tab.to_string())

in-memory patch removed; original sim.simulator._routes restored: True
                             generated  delivered  unmet  shipping  waiting    total  recovery_days
S0                                60.0       60.0    0.0     290.0      0.0    290.0            2.0
S1                                60.0       60.0    0.0     360.0      0.0    360.0            1.0
S2                                60.0       55.0    5.0     355.0   4560.0   5395.0            NaN
plan=None, no closure (S0)        60.0       60.0    0.0     270.0   1824.0   2094.0            5.0
plan=None, H1→D closed (S1)       60.0        0.0   60.0      38.0  54336.0  60134.0            NaN


### 7b. Test results against the Day 6 validation list  `[TOY]`
Each row is an executable assertion. A pass here shows the **interface is mathematically consistent on a toy graph**; it does **not** validate the real 292-commodity network or operational realism.

In [9]:
if REPO:
    R = results; D = SCN["demand_days"]; out = []
    def check(name, cond, detail=""): out.append({"test": name, "pass": bool(cond), "detail": detail}); assert cond, name

    # T1 single-OD compatibility
    check("T1 single-OD compatibility (legacy dict == nested plan)", legacy == nested, f"routes={legacy}")
    # T2 multi-OD handled + flow conservation (simulate() asserts conservation every day; we also check totals)
    for n in plans:
        r = R[n]; gen = D * sum(G.graph["demands"].values())
        check(f"T2 {n}: generated = delivered + unmet", abs(r["generated"] - gen) < 1e-9 and abs(r["generated"] - r["delivered"] - r["unmet"]) < 1e-6,
              f"gen={r['generated']:.1f} deliv={r['delivered']:.1f} unmet={r['unmet']:.1f}")
    # T3 capacity: plan respects shared hub capacity (I5) and sim stays conservative when capacity is halved
    check("T3a LP plans satisfy shared hub capacity (I5)", all(validate_plan(G, p) for p in plans.values()))
    check("T3b halving capacity: still conserved, cost rises (queues form)", abs(stress["generated"] - stress["delivered"] - stress["unmet"]) < 1e-6 and stress["waiting_cost"] > R["S0"]["waiting_cost"],
          f"waiting {R['S0']['waiting_cost']:.0f} -> {stress['waiting_cost']:.0f}; recovery_days {stress['recovery_days']}")
    # T4 closed lanes: plan on closed lane is rejected; compliant plan unaffected by the closure
    bad_p = copy.deepcopy(plans["S1"]); bad_p["flows"][("A","D")] = {("A","H1"): 6.0, ("H1","D"): 6.0}
    try: validate_plan(G, bad_p); caught = False
    except ValueError: caught = True
    check("T4 flow on a closed lane is rejected by validate_plan (I6)", caught)
    # T5 unmet demand
    check("T5 S2: sim unmet == LP unmet × demand_days", abs(R["S2"]["unmet"] - sum(plans["S2"]["unmet"].values()) * D) < 1e-6, f"sim={R['S2']['unmet']:.2f} LP={sum(plans['S2']['unmet'].values())*D:.2f}")
    # T6 LP↔simulator cost parity on the part both model: shipping (+ penalty)
    for n in plans:
        lp_ship = plans[n]["travel"] * D
        check(f"T6 {n}: sim shipping == LP travel × demand_days", abs(R[n]["shipping_cost"] - lp_ship) < 1e-6, f"sim={R[n]['shipping_cost']:.3f} LP={lp_ship:.3f}")
    # T7 no-disruption baseline: everything delivered, no waiting with the capacity-aware plan
    check("T7 no-disruption: S0 plan delivers everything with zero waiting", R["S0"]["unmet"] < 1e-9 and R["S0"]["waiting_cost"] == 0, f"waiting={R['S0']['waiting_cost']}")
    # T8 recovery under disruption: LP plan beats doing nothing when H1→D closes
    check("T8 closure S1: LP plan total << plan=None total", R["S1"]["total_cost"] < 0.05 * R["plan=None, H1→D closed (S1)"]["total_cost"], f"{R['S1']['total_cost']:.0f} vs {R['plan=None, H1→D closed (S1)']['total_cost']:.0f}")
    # T9 plan=None is NOT a neutral baseline even with no failure (G13)
    check("T9 plan=None, no closure, congests (waiting > 0) while capacity-aware S0 plan does not",
          R["plan=None, no closure (S0)"]["waiting_cost"] > 0 and R["S0"]["waiting_cost"] == 0, f"waiting {R['plan=None, no closure (S0)']['waiting_cost']:.0f} vs 0")
    # G8 horizon dependence
    check("G8 waiting cost of permanently-unmet legs scales with horizon (total_cost is horizon-dependent)", hz[20] < hz[40] < hz[80], f"waiting @20/40/80 days = {hz[20]:.0f}/{hz[40]:.0f}/{hz[80]:.0f}")

    # T10 independent cross-check with YOUR OWN static evaluator sim/simulate.py::simulate_scenario (also accepts the nested plan)
    from sim.simulate import simulate_scenario
    def gd(G, Dcap=None):
        nc = {n: G.nodes[n]["cap"] for n in G}
        if Dcap is not None: nc["D"] = Dcap
        return {"node_cap": nc, "lane_cost": {e: G.edges[e]["cost"] for e in G.edges}, "demand": dict(G.graph["demands"])}
    for n, p in plans.items():
        r = simulate_scenario(gd(G, 1e9), {"closed_lanes": SCEN[n], "penalty": PENALTY}, p["flows"])
        check(f"T10 {n}: simulate_scenario shipping/unmet == LP (destination cap lifted)", abs(r["shipping"] - p["travel"]) < 1e-6 and abs(r["unmet"] - sum(p["unmet"].values())) < 1e-6,
              f"shipping {r['shipping']} vs {p['travel']}; unmet {r['unmet']}")
    r0 = simulate_scenario(gd(G), {"closed_lanes": [], "penalty": PENALTY}, plans["S0"]["flows"])
    check("G10 simulate_scenario with destination cap = 0 reports ALL legs unmet (semantic mismatch with LP / queue sim)", abs(r0["unmet"] - 12.0) < 1e-9, f"unmet={r0['unmet']} while LP unmet=0")



    # T11: explicit `closed_lanes` contract field is enforced.
    closed_bad = copy.deepcopy(plans["S1"]); closed_bad["closed_lanes"] = [("A", "D")]
    try: validate_plan(G, closed_bad); caught = False
    except ValueError: caught = True
    check("T11 closed_lanes field rejects a flow on a listed closed edge", caught)

    # T12: non-finite unmet is rejected (NaN must not slip through comparisons).
    nan_bad = copy.deepcopy(plans["S0"]); nan_bad["unmet"][("A", "D")] = float("nan")
    try: validate_plan(G, nan_bad); caught = False
    except ValueError: caught = True
    check("T12 non-finite unmet is rejected", caught)

    # T13: edge capacity is shared across commodities and checked by the contract validator.
    edge_limited = G.copy(); edge_limited.edges["H1", "D"]["cap"] = 7.0
    try: validate_plan(edge_limited, plans["S0"]); caught = False
    except ValueError: caught = True
    check("T13 shared edge capacity violation is rejected", caught)

    # T14: a balanced but cyclic commodity flow is rejected by the real route decomposer.
    cycle_graph = G.copy(); cycle_graph.nodes["H1"]["cap"] = 9.0; cycle_graph.add_edge("H1", "B", cost=1.0, cap=math.inf)
    cycle_bad = copy.deepcopy(plans["S0"])
    cycle_bad["flows"][("A", "D")][("H1", "B")] = 1.0
    cycle_bad["flows"][("A", "D")][("B", "H1")] = 1.0
    try: validate_plan(cycle_graph, cycle_bad); caught = False
    except ValueError: caught = True
    check("T14 balanced cycle is rejected by existing route decomposition", caught)

    # T15: zero demand is a valid explicit OD with no allocated flow.
    zero_graph = G.copy(); zero_graph.graph["demands"][("A", "D")] = 0.0
    zero_plan = copy.deepcopy(plans["S0"]); zero_plan["demands"][("A", "D")] = 0.0
    zero_plan["flows"][("A", "D")] = {}; zero_plan["unmet"][("A", "D")] = 0.0
    check("T15 zero-demand OD is valid", validate_plan(zero_graph, zero_plan))

    # T16: failed origin dispatch is blocked and queued demand drains after recovery.
    check("T16 failed hub blocks dispatch during failure and conserves demand",
          failure_result["history"][0]["outside_backlog"] == 2.0
          and failure_result["history"][0]["in_transit"] == 0.0
          and failure_result["delivered"] == 2.0 and failure_result["unmet"] == 0.0,
          f"day0={failure_result['history'][0]}, delivered={failure_result['delivered']}")

    # T17: zero-capacity destination hub can receive a final delivery in the queue simulator.
    zero_dest = nx.DiGraph(); zero_dest.add_node("A", cap=2.0); zero_dest.add_node("D", cap=0.0)
    zero_dest.add_edge("A", "D", cost=1.0); zero_dest.graph["demands"] = {("A", "D"): 2.0}
    zd = S.simulate(zero_dest, dict(demand_days=1, horizon=4, failure_days=0), None)
    check("T17 zero-capacity destination receives final deliveries", zd["delivered"] == 2.0 and zd["unmet"] == 0.0)

    print(pd.DataFrame(out).to_string(index=False))
    print(f"\n{sum(o['pass'] for o in out)}/{len(out)} toy checks passed")
else:
    print("repo not found - skipped")

                                                                                                         test  pass                                                                                                                   detail
                                                      T1 single-OD compatibility (legacy dict == nested plan)  True                                                                                               routes=[(['A', 'D'], 6.0)]
                                                                         T2 S0: generated = delivered + unmet  True                                                                                            gen=60.0 deliv=60.0 unmet=0.0
                                                                         T2 S1: generated = delivered + unmet  True                                                                                            gen=60.0 deliv=60.0 unmet=0.0
                                                    

**What these results do and do not show**

- The proposed adapter calls the queue simulator's real single-OD decomposer once per OD; it is patched in memory and restored in `finally`. This is a toy demonstration, not production integration.
- The production optimizer still discards its per-commodity solution, and the production queue simulator still rejects a planned multi-OD flow map. Both owner-reviewed changes are required for end-to-end integration.
- Shipping parity applies only to travel cost on the toy graph. Queue waiting cost and horizon-dependent waiting for unmet traffic are separate; do not claim total-cost parity.
- `plan=None` chooses shortest-cost routes without planning them around shared capacity or closures, so it is not a capacity-aware do-nothing baseline.
- The real-data optimizer demand is 5,395/14 = 385.3571 legs/day; the queue test's in-window departure demand is 5,357/14 = 382.6429. Resolve the 38-leg difference or use one agreed explicit demand vector before a real comparison.
- Code uses a 96-hour unmet penalty, `docs/c_notes.md` says 48 hours, and the queue default is 20. The toy run pins 96 only for its stated scenario; no project-wide canonical value is established.
- The optimizer accepts one lane closure; the queue has no closure scenario field. The toy closure was permanent for that run. Hub-failure optimization is unsupported.
- The static evaluator and queue simulator differ for zero-capacity destination-only hubs. Queue buffers remain assumed storage, not measured receiving throughput.
- A previous draft's closure assertion used the wrong field name (`closed` instead of the documented `closed_lanes`); do not cite its 19/19 count. The corrected closure and added edge-capacity/non-finite-unmet checks are in this notebook's executed results.
- No real-network or operational-realism validation was performed.


## 8. Smallest production changes and open decisions `[PROPOSED]` / `[OPEN]`

| Change | File | Owner | Scope / side effect |
|---|---|---|---|
| Expose per-OD solution variables, unmet rates, travel term, demand vector, closure, units, and provenance in a new `solve_multiflow_plan`; retain `solve_multiflow` as a compatibility wrapper returning its existing two values | `opt/c_multiflow.py` | Person C | API addition; keep LP objective and existing scalar caller behavior unchanged |
| Accept the proposed multi-OD nested map or OD-edge-flow records in `_routes`, invoking the existing single-OD decomposition once per OD; retain legacy single-OD dict behavior | `sim/simulator.py` | Person A/Jahnavi | Queue route-input change; requires single-OD and multi-OD owner regression tests |
| Keep validation and toy evidence in this authoritative notebook for now; no separate Day 6 notebook or executed copy | B06 / `experiments/day6/` | Person B | No production interface is installed |

**Patch shape for owner review (not applied):**

```python
# opt/c_multiflow.py
# Factor the existing LP solve so the new helper can serialize res.x by OD.
def solve_multiflow_plan(G, closed=None):
    ...  # return demands, per-OD flows, unmet, travel, objective, units, provenance

def solve_multiflow(G, closed=None):
    plan = solve_multiflow_plan(G, closed=closed)
    return plan["total"], plan["unmet_total"]

# sim/simulator.py
# Keep plan=None and the one-OD mapping branches unchanged.
# For a multi-OD map/record list, normalize once, then reuse the existing
# single-OD route decomposition for each OD and concatenate its route jobs.
```

These changes need review and explicit approval from the owners before implementation. Do not run the real 292-lane LP until the demand, penalty, closure, and failure semantics are agreed and toy/interface checks are accepted.

| Decision | Still open |
|---|---|
| Demand source | 5,395/14 optimizer `trips`, or 5,357/14 in-window departures |
| Unmet penalty | 96 h in code/tests, 48 h in `docs/c_notes.md`, queue default 20 |
| Closure semantics | One static lane as supported by optimizer, or multiple/time-varying closures requiring a broader interface |
| Hub failure | Whether the optimizer must anticipate failed-hub scenarios, and how receiving versus dispatch is modeled |
| Zero-capacity destination | Whether the static evaluator or daily queue simulator semantics govern operational plan evaluation |

The Day 6 objective is optimizer–simulator integration. No live remote fetch was performed.


## 9. Validation status and completion

The authoritative notebook's corrected toy checks cover plan-demand agreement, multi-OD flow conservation, delivered-plus-unmet accounting, shared hub and edge capacities, closed lanes, non-finite unmet values, cycle rejection using the existing route decomposer, single-OD compatibility, zero demand, failed-hub queue behavior, and zero-capacity final destinations. The existing single-OD toy script was also run with output redirected to `experiments/day6/`.

| Acceptance area | Status |
|---|---|
| Single-OD behavior | `[TOY]` passed |
| Multi-OD toy adapter | `[TOY]` passed; in-memory only |
| Plan invariants | `[TOY]` 26/26 passed |
| Existing toy queue test | `[TOY]` 21 scenarios, 3 comparisons, assertions completed |
| Production optimizer plan export | **Not implemented**; owner review needed |
| Production queue multi-OD input | **Not implemented**; owner review needed |
| Agreed demand, penalty, closure, and failure semantics | `[OPEN]` |
| Real 292-lane optimization/evaluation | Not run; unresolved assumptions and owner approval required |

**Final Day 6 status: PARTIALLY COMPLETE.** Toy validation is not real-network validation and does not establish operational performance.
